# LA Wildfire Recovery — From Agenda Setting Comments to Actionable Policies For Deliberation
This notebook turns the many free-text wildfire-recovery comments obtained during agenda setting into a set of actionable policies/options for the deliberation stage

1. **Filter & Summarise** – Use Snowflake Cortex (`AI_FILTER`, `AI_COMPLETE`) to keep only action-oriented statements and condense them into succinct, imperative “options.”  
2. **Embed** – Generate 1 024-D sentence embeddings (`EMBED_TEXT_1024`) with a common wildfire context primer.  
3. **Model Topics** – Dimensionality-reduce (PCA → UMAP) and cluster (HDBSCAN, BERTopic) with Optuna-driven hyper-parameter tuning for optimal topic separation.  
4. **Label & Describe** – Call Cortex again to craft a plain-English topic name, neutral summary, and example actions.  
5. **Persist** – Save comment-level and topic-level outputs back to Snowflake for dashboards or downstream analysis.  


## Find Relevant Comments
This query pulls wildfire-related comments and keeps only those that *Snowflake Cortex*’s `AI_FILTER()` large-language-model (LLM) classifier flags as **action-oriented ideas about rebuilding or long-term community resilience**. All other rows are discarded.

### LLM-powered filtering with `AI_FILTER()`  
* **`AI_FILTER()`** evaluates a natural-language condition and returns `TRUE`/`FALSE` per row, letting you filter text with a single Boolean predicate.  https://docs.snowflake.com/en/sql-reference/functions/ai_filter
* The first call wraps the comment text (`CONTENT`) with a lead-in sentence so the model knows it is assessing a *wildfire survivor’s suggestion/problem*.  
* The second call feeds a *structured prompt object* (built with `PROMPT()`) that asks only **“YES or NO”** whether the statement contains a concrete rebuilding or planning idea.


In [ ]:
select * from ANALYTICS_ENGCA_PRD.ETHELO.COMMENTS
where 
AI_FILTER(CONCAT('This statement from a person affected by a recent wildfire addressed to their state governement contains a suggestion, proposal for action, or a clearly stated problem: ', CONTENT)) 
and  TARGET != 'Emotional & mental health support' and Reply_to_ID is null 
and AI_FILTER(
        PROMPT(
          '**Question (answer YES or NO only):**
        Does the following statement contribute a specific *idea, concern, or suggestion* related to urgent rebuilding of homes, businesses, places of worship, or public spaces **OR** long-term planning for safer, more affordable, more resilient communities after the wildfire in Los Angeles?. Statement: "{0}"',
          CONTENT                   -- fills {0}
        )
      );


## Generate succinct policy **options** from each comment  
This cell turns every shortlisted wildfire-recovery comment into one or more **imperative, policy-ready options** using a generative AI model and a bit of string wrangling.

### Step 1  `possible_option_comments`  
Re-selects the rows we filtered in the previous step so we have a clean starting point (three columns plus the `comment_id`).

### Step 2  `options_per_comment` — LLM summarisation  
* **`AI_COMPLETE(model, prompt)`** calls the DeepSeek-R1 model to _rewrite_ each full comment into a terse imperative statement (e.g., “**Waive permitting fees**”).
* The prompt is built on-the-fly with **`PROMPT()`**, which lets us embed the `TARGET` tag and the comment `CONTENT` into a templated instruction containing numbered placeholders like `{0}`. :contentReference[oaicite:1]{index=1}  
* **`SPLIT_PART(…, '</think>\n\n', 2)`** trims away the model’s hidden chain-of-thought, keeping only the answer after the special `</think>` marker. :contentReference[oaicite:2]{index=2}  

### Step 3  `filtered_options_per_comment` — quality gate  
Rows where the model produced an empty string (≤ 10 chars after trimming) are dropped, keeping only substantive options.

### Final SELECT — explode the list  
* **`SPLIT_TO_TABLE(string, ';')`** flattens the semicolon-separated list of options into one row per idea.
* `TRIM()` cleans up stray leading/trailing spaces.

> **Result:** a tidy, one-option-per-row table—`comment_id`, topic `target`, original `content`, and the distilled **option**—ready for topic modeling downstream


In [ ]:
with possible_option_comments as (
    select 
        comment_id
        ,target
        ,content 
    from {{possible_option_comments}} 
),

options_per_comment as (
    select 
        comment_id
        ,target
        ,content 
        ,SPLIT_PART( AI_COMPLETE('deepseek-r1', concat('''The following is a statement from a person affected by a recent wildfire and is addressed to their state governement. If this statement contains a suggestion or proposal for action by government, then summarize that proposal into a succinct imperative statement starting with a verb. The succinct imperative statement should focus on the core policy intent and avoid unecessary language or details that make it less generalizable. If the statement contains more than one suggestion or proposal for action, make an imperative statement for each one and separate them into a semi-colon separate list. If there are no suggestions or proposals, return an empty string. Ignore any proper names such as cities, streets, or officials and focus on the core policy intent. IMPORTANT: Your response should contain the imperative statements and nothing else. --- STATEMENT --- Regarding the wildfire recovery topic of ''',target, ''' the person said the following: "''',CONTENT,'"')), '</think>\n\n', 2) as options
    from possible_option_comments
),
filtered_options_per_comment as (
    select 
        comment_id
        ,target
        ,content 
        ,options
    from options_per_comment
    where length(options) > 10
) 
SELECT
    comment_id
    ,target
    ,content
    ,TRIM(v.value) AS option
FROM filtered_options_per_comment AS t,
     TABLE( SPLIT_TO_TABLE(t.options, ';') ) as v 

## Build embeddings for **topic modeling**  
This step converts each policy option into a high-dimensional vector that will feed our downstream topic-modeling workflow.

### What happens in this cell  
1. **Table creation** – `CREATE OR REPLACE … option_embeddings` makes reruns cheaper and easier. 
2. **Context primer** – A short, shared wildfire-recovery preamble is prepended to every `option` so the embedding model views all proposals through the same lens.  
3. **Embedding generation** – `SNOWFLAKE.CORTEX.EMBED_TEXT_1024('voyage-multilingual-2', …)` returns a 1024-dim vector (`primed_comment_embedding`). These vectors will later be reduced (e.g., via UMAP) and clustered (e.g., HDBSCAN/BERTOPIC) to discover thematic groupings.  
4. **Quality filter** – Options ≤ 10 characters are skipped; they lack meaningful semantic signal for topic modeling.  
5. **Preview** – The final `SELECT *` shows the new table: IDs, raw text, primed text, and embeddings.

> **Outcome:** we now have a vector representation for every actionable suggestion, ready for dimensionality reduction and clustering to reveal the main themes in community feedback.


In [ ]:
Create or replace table TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.option_embeddings as

Select
    comment_id
    ,target
    ,content
    ,option
    -- Create text to be used for embedding. We construct a conditional context primer by combining: 
    -- 1) an universal introduction
    -- 2) the option
    ,CONCAT(
        -- Part 1: Universal Wildfire Context Primer
        'This proposed government action comes from a resident engaged in post-wildfire recovery after the January 2025 Southern California fires (Palisades, Eaton, and nearby events). With thousands of structures lost and communities displaced, officials and citizens are debating how to rebuild homes, businesses, infrastructure, and public spaces while simultaneously strengthening future resilience, affordability, environmental health, and social equity. The following directive is one concrete policy proposal: ', option
    ) AS primed_comment_for_embedding
    ,SNOWFLAKE.CORTEX.EMBED_TEXT_1024('voyage-multilingual-2', primed_comment_for_embedding ) as primed_comment_embedding
from {{options_per_comment}}
where length(option) > 10;

Select * from TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.option_embeddings

## Load embeddings into Python  
* Sets cache dirs (`/tmp`) to avoid write issues.  
* Pulls `option_embeddings` into a Pandas DataFrame via Snowpark.  
* Splits out:  
  * `full_dim_embedding` – 1 024-D vectors  
  * `docs` – plain-text options  
Ready for UMAP + BERTopic in the next step.

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
import numpy as np
import os

#change environment variables to get bertopic to work in this environment
os.environ["NUMBA_CACHE_DIR"] = "/tmp"
os.environ["TRANSFORMERS_CACHE"] = "/tmp"
os.environ["TRANSFORMERS_CACHE"] = "/tmp"

from snowflake.snowpark.context import get_active_session
session = get_active_session()

cmnt_embeddings_df = session.sql('''
Select
    comment_id
    ,target
    ,content
    ,option     
    ,PRIMED_COMMENT_EMBEDDING
FROM TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.option_embeddings
''').to_pandas()

# cmnt_embeddings_df = option_embeddings.to_pandas()

full_dim_embedding = np.array(cmnt_embeddings_df['PRIMED_COMMENT_EMBEDDING'].tolist())
full_embedding = np.array(cmnt_embeddings_df['PRIMED_COMMENT_EMBEDDING'].tolist())

docs = np.array(cmnt_embeddings_df['OPTION'].tolist())


## Dimensionality-reduce embeddings with PCA  
We shrink the 1024-D vectors before UMAP/HDBSCAN to speed things up.

* **`apply_pca_with_variance_threshold()`** – custom helper that  
  * (optionally) standardizes features,  
  * picks the smallest component count that captures a chosen share of variance,  
  * returns the reduced matrix plus the fitted `PCA` & `scaler`.  
* Here we keep **98 %** of variance, dropping the embedding dimensionality to _n_components_.  
  The resulting `full_embedding` is what we’ll pass to UMAP/BERTopic next.

In [ ]:
#do pca first to speed up optimization
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def apply_pca_with_variance_threshold(embeddings, min_variance_explained=0.95, standardize=True):
    """
    Apply PCA to embeddings with a minimum variance explained threshold.
    
    Parameters:
    -----------
    embeddings : np.ndarray
        Input embeddings array of shape (n_samples, n_features)
    min_variance_explained : float, default=0.95
        Minimum cumulative variance to be explained by the components (0.0 to 1.0)
    standardize : bool, default=True
        Whether to standardize features before PCA
    
    Returns:
    --------
    reduced_embeddings : np.ndarray
        PCA-transformed embeddings with reduced dimensions
    n_components : int
        Number of components selected
    variance_explained : float
        Actual cumulative variance explained by selected components
    pca_model : PCA
        Fitted PCA model for future transforms
    scaler : StandardScaler or None
        Fitted scaler if standardization was used
    """
    
    print(f"Original embedding shape: {embeddings.shape}")
    
    # Standardize features if requested
    scaler = None
    if standardize:
        scaler = StandardScaler()
        embeddings_scaled = scaler.fit_transform(embeddings)
    else:
        embeddings_scaled = embeddings.copy()
    
    # First, fit PCA with all components to see variance explained
    pca_full = PCA()
    pca_full.fit(embeddings_scaled)
    
    # Calculate cumulative variance explained
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    
    # Find number of components needed for minimum variance
    n_components = np.argmax(cumulative_variance >= min_variance_explained) + 1
    
    # Ensure we don't exceed the maximum possible components
    max_components = min(embeddings.shape[0], embeddings.shape[1])
    n_components = min(n_components, max_components)
    
    # If we can't reach the threshold, use all available components
    if cumulative_variance[-1] < min_variance_explained:
        n_components = max_components
        actual_variance = cumulative_variance[-1]
        print(f"Warning: Maximum possible variance explained is {actual_variance:.4f}, "
              f"which is less than requested {min_variance_explained:.4f}")
    else:
        actual_variance = cumulative_variance[n_components - 1]
    
    # Fit PCA with selected number of components
    pca = PCA(n_components=n_components)
    reduced_embeddings = pca.fit_transform(embeddings_scaled)
    
    print(f"Reduced embedding shape: {reduced_embeddings.shape}")
    print(f"Number of components: {n_components}")
    print(f"Variance explained: {actual_variance:.4f} ({actual_variance*100:.2f}%)")
    print(f"Dimensionality reduction: {embeddings.shape[1]} -> {n_components} "
          f"({n_components/embeddings.shape[1]*100:.1f}% of original)")
    
    return reduced_embeddings, n_components, actual_variance, pca, scaler

# Apply PCA with 95% variance explained (you can adjust this threshold)
min_variance_threshold = 0.98  # Adjust this value as needed (0.90, 0.95, 0.99, etc.)

full_embedding, n_components, variance_explained, pca_model, scaler = apply_pca_with_variance_threshold(
    full_dim_embedding, 
    min_variance_explained=min_variance_threshold,
    standardize=True
)


## Prep: scoring helpers for Optuna search  
This cell lays the groundwork for automated hyper-parameter tuning.

* **Imports** Optuna, BERTopic, UMAP, HDBSCAN, and clustering metrics.  
* **Silences Optuna logs** for cleaner output.  
* **Scoring utilities**  
  * `score_topic_count()` – rewards models that produce ~55 topics (our target granularity).  
  * `score_classified_percentage()` – favors runs where 75–95 % of documents receive a topic label.  
  * Normalizers convert **silhouette** and **Calinski-Harabasz** scores to the 0-1 range.  
These helpers feed the Optuna objective defined in the next cell.

In [ ]:
import time
import optuna
import numpy as np
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.feature_extraction.text import CountVectorizer

# Disable printing a message for each trial
optuna.logging.set_verbosity(optuna.logging.WARNING)


# Returns higher values when the input number of topics matches what we
# want/expect from our data. This should be configured per comment category!
def score_topic_count(num_topics: int) -> float:
    if num_topics == 55:
        return 1 # we want this many topics
    elif  num_topics >= 54 and num_topics <= 56:
        return 1 # this is good, but slightly less desireable
    elif  num_topics >= 53 and num_topics <= 57:
        return 0.99 # this is good, but slightly less desireable
    elif  num_topics >= 52 and num_topics <= 58:
        return 0.98 # this is good, but slightly less desireable
    elif  num_topics >= 50 and num_topics <= 60:
        return 0.9 # this is good, but slightly less desireable
    elif  num_topics >= 45 and num_topics <= 65:
        return 0.8 # this is good, but slightly less desireable
    # elif num_topics in (2,6):
    #     return .5 # this is okay, but less desireable
    else:
        return 0


# Returns higher values when the number of classified documents is relatively high
# (but not too high)
def score_classified_percentage(classified_ratio: float) -> float:
    if .75 <= classified_ratio <= .95: # >90% classified means something went wrong, probably
        return 1.0
    elif .6 <= classified_ratio < .75:
        return .8
    elif .4 <= classified_ratio < .55:
        return 0
    elif .85 <= classified_ratio < 1:
        return .85
    # elif .9 <= classified_ratio <= 1:
    #     return .5
    else:
        return 0


# Normalize from (-1, 1) to (0, 1), where 1 is good
def normalize_silhouette_score(ss):
    return (ss + 1) / 2


# Normalize from (0, inf) to (0, 1). We use a sigmoid function to cap values at 1.
# Using a constant value of 5 is somewhat arbitrary, though most input score values
# are <10, so anything much higher might compress output values too much.
def normalize_calinski_harabasz_score(chs, C=5):
    return chs / (chs + C)     
        

## Optuna loop: find the best BERTopic settings  
Here we launch a 300-trial Optuna study that searches UMAP + HDBSCAN hyper-parameters for BERTopic.

* **`objective(trial)`**  
  1. Samples UMAP (neighbors, components, min_dist) and HDBSCAN (min_cluster_size, min_samples, ε) settings.  
  2. Builds a BERTopic instance with those parameters.  
  3. Fits it to our `docs` + `full_embedding`.  
  4. Scores the result:  
     * Cluster quality → silhouette + Calinski-Harabasz (normalized).  
     * Heuristics → topic-count score & classified-percentage score.  
     * Final fitness = quality × heuristics.  
* **`study.optimize()`** runs the search, then prints the best score and the hyper-parameters you can copy into the production config block for validation.  
Execution time and trial count are configurable via `n_trials`.


In [ ]:
def objective(trial: optuna.trial.Trial):    
    # Set up parameters to optimize for
    umap_n_neighbors = trial.suggest_int('umap_n_neighbors', 25, 45)
    umap_n_components = trial.suggest_int('umap_n_components', 5, 15)
    umap_min_dist = trial.suggest_float('umap_min_dist', 0.0, 0.2)

    hdbscan_min_cluster_size = trial.suggest_int('hdbscan_min_cluster_size', 3, 5)
    hdbscan_min_samples = trial.suggest_int('hdbscan_min_samples', 2, 5)
    hdbscan_cluster_selection_epsilon = trial.suggest_float('hdbscan_cluster_selection_epsilon', 0.0, 0.2) # Note: only valid if method is 'eom'

    # Create models to execute with
    umap_model = UMAP(n_neighbors=umap_n_neighbors,
                      n_components=umap_n_components,
                      min_dist=umap_min_dist,
                      metric='cosine',
                      random_state=42) # For reproducibility within this trial

    hdbscan_model = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size,
                            min_samples=hdbscan_min_samples,
                            cluster_selection_method='eom',
                            prediction_data=True,
                            gen_min_span_tree=True,
                            cluster_selection_epsilon=hdbscan_cluster_selection_epsilon)

    # This is the same vectorizer model we use when actually running topic modeling
    vectorizer_model = CountVectorizer(
        min_df=0.01,
        max_df=0.5,
        ngram_range=(1, 1)
    )

    # Finally initialize topic modeler
    topic_model = BERTopic(
        top_n_words=1,
        hdbscan_model=hdbscan_model,
        umap_model=umap_model,
        vectorizer_model=vectorizer_model,
        verbose=False
    )


    # Extract documents and embeddings for this category
    # category_docs_full = np.array(docs)[category_indices]
    category_docs = np.array(docs) # Ensure docs is indexable like a list or numpy array
    category_embeddings = full_embedding
    num_documents = len(category_docs)

    # Convert to numpy array to make BERTopic happy
    category_embeddings = np.array(category_embeddings)

    try:
        topics, _ = topic_model.fit_transform(list(category_docs), category_embeddings)
        topic_info = topic_model.get_topic_info()
    except Exception as e:
        # It's not uncommon to try and optimize invalid parameters for a given dataset. The best we
        # can do here is return a strong negative signal.
        return -2



    mask = [value != -1 for value in topics]
    assigned_fe = category_embeddings[mask]
    assigned_tp = np.array(topics)[mask]

    # Calculate clustering evaluation scores
    ss = silhouette_score(assigned_fe, assigned_tp)
    ss = normalize_silhouette_score(ss)

    chs = calinski_harabasz_score(assigned_fe, assigned_tp)
    chs = normalize_calinski_harabasz_score(chs)

    # Combine clustering evaluation scores into a "base" score
    base_score = (ss + chs) / 2
    
    # Calculate score scaling factors
    num_topics = len(topic_info[topic_info.Topic != -1])
    topic_count_score_val = score_topic_count(num_topics)

    num_classified_documents = np.sum(np.array(topics) != -1)
    classified_ratio = (num_classified_documents / num_documents)
    classified_percentage_score_val = score_classified_percentage(classified_ratio)

    final_score = base_score * topic_count_score_val * classified_percentage_score_val

    return final_score


start_time = time.time()
n_trials = 300
study = optuna.create_study(direction='maximize', study_name="bertopic_optimization")

study.optimize(objective, n_trials=n_trials)

print()
print(f'elapsed: {round(time.time()-start_time, 2)} sec')

print(f"Best trial number: {study.best_trial.number}")
print(f"Best value: {study.best_value:.4f}")
print("Best hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# elapsed: 1206.24 sec  
# Best trial number: 63  
# Best value: 0.6759  
# Best hyperparameters:  
#   umap_n_neighbors: 39  
#   umap_n_components: 10  
#   umap_min_dist: 0.08979994025437738  
#   hdbscan_min_cluster_size: 5  
#   hdbscan_min_samples: 4  
#   hdbscan_cluster_selection_epsilon: 0.04171367112388551

## Train BERTopic with best-found hyper-parameters  
* Hard-codes the Optuna-selected UMAP & HDBSCAN values (or adjust manually).  
* Fits **BERTopic** on `category_docs` + their embeddings → returns `topics` and confidences.  
* Builds `doc_df` linking each `COMMENT_ID` to its topic and full text, then prints topic counts and outlier share.


In [ ]:
umap_n_neighbors = 39 #study.best_params["umap_n_neighbors"]
umap_n_components = 10 #study.best_params["umap_n_components"]
umap_min_dist = 0.08979994025437738   #study.best_params["umap_min_dist"]

hdbscan_min_cluster_size = 5 #study.best_params["hdbscan_min_cluster_size"]
hdbscan_min_samples = 4 #study.best_params["hdbscan_min_samples"]
hdbscan_cluster_selection_epsilon = 0.04171367112388551 #study.best_params["hdbscan_cluster_selection_epsilon"] 

# umap_n_neighbors = study.best_params["umap_n_neighbors"]
# umap_n_components = study.best_params["umap_n_components"]
# umap_min_dist = study.best_params["umap_min_dist"]

# hdbscan_min_cluster_size = study.best_params["hdbscan_min_cluster_size"]
# hdbscan_min_samples = study.best_params["hdbscan_min_samples"]
# hdbscan_cluster_selection_epsilon = study.best_params["hdbscan_cluster_selection_epsilon"]

# Extract documents and embeddings 
category_docs = np.array(docs) # Ensure docs is indexable like a list or numpy array
category_embeddings = full_embedding

# Convert to numpy array to make BERTopic happy
category_embeddings = np.array(category_embeddings)

# Create a custom CountVectorizer with stop words removal and document frequency parameters
vectorizer_model = CountVectorizer(
    stop_words='english',           # Remove English stop words
    min_df=0.005,                    # Ignore terms that appear in less than X% of documents
    max_df=0.5,                     # Ignore terms that appear in more than X% of documents
    ngram_range=(1, 2)              # Consider both unigrams and bigrams for more meaningful phrases
)

# Create models to execute with
umap_model = UMAP(n_neighbors=umap_n_neighbors,
                  n_components=umap_n_components,
                  min_dist=umap_min_dist,
                  metric='cosine',
                  random_state=42) # For reproducibility within this trial

hdbscan_model = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size,
                        min_samples=hdbscan_min_samples,
                        cluster_selection_method='eom',
                        prediction_data=True,
                        gen_min_span_tree=True,
                        cluster_selection_epsilon=hdbscan_cluster_selection_epsilon)
    
# Create BERTopic model
topic_model = BERTopic(
    top_n_words=10,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    verbose=False
)


topics, probs = topic_model.fit_transform(list(category_docs), category_embeddings) # Pass docs as list


    
# Create topic info dataframe
topic_info = topic_model.get_topic_info()
print(f"Found {len(topic_info) -1 if not topic_info.empty and topic_info.iloc[0]['Topic'] == -1 else len(topic_info)} actual topics (excluding outliers)")




# Create dataframe with documents and their topics
doc_df = pd.DataFrame({
    'COMMENT_ID': cmnt_embeddings_df['COMMENT_ID'],
    'SUMMARY': category_docs,
    'TOPIC': topics,
    'CONTENT': cmnt_embeddings_df['CONTENT']
})
    

# Count how many documents were assigned to each topic
unique_topic_ids, topic_doc_counts = np.unique(topics, return_counts=True)
#print("Topic distribution:")
for topic_id, count in zip(unique_topic_ids, topic_doc_counts):
    if topic_id != -1:
        print(f"  Topic {topic_id}: {count} documents")
if -1 in unique_topic_ids:
    print(f"  Unassigned (-1): {list(topics).count(-1)} documents")
else:
    print(f"  Unassigned (-1): 0 documents")





## Summarise topics with Snowflake Cortex  
* **`generate_topic_descriptions()`**  
  1. Drops outliers (`TOPIC = -1`) and gathers all original comments per topic.  
  2. Builds a prompt that includes the topic’s top keywords + its comments.  
  3. Calls **Cortex `complete`** (`claude-3-5-sonnet`) to return three `^`-separated pieces:  
      * 3–7-word imperative topic name  
      * 50–100-word neutral summary (pros + trade-offs)  
      * comma-separated list of example actions  
  4. Collects the AI output in `topic_descriptions_df`.

* Prints the generated names, summaries, and actions, then merges them back into `topic_info_with_desc` for downstream reporting.


In [ ]:
# Add this code after your existing BERTopic model creation and before writing to Snowflake

from snowflake.cortex import complete, CompleteOptions
import pandas as pd

# Create topic aggregation similar to your SQL
def generate_topic_descriptions(doc_df, topic_info, topic_model, session):
    """
    Generate topic descriptions using Snowflake Cortex in Python
    """
    
    # Filter out outliers (topic -1)
    filtered_doc_df = doc_df[doc_df['TOPIC'] >= 0].copy()
    
    # Aggregate comments by topic
    topic_agg = []
    
    for topic in filtered_doc_df['TOPIC'].unique():
        topic_docs = filtered_doc_df[filtered_doc_df['TOPIC'] == topic]
        
        # Get topic representation from topic_model
        try:
            topic_words = topic_model.get_topic(topic)
            if topic_words:
                representation = ', '.join([word for word, _ in topic_words])
            else:
                representation = ""
        except:
            representation = ""
        
        # Aggregate comments (truncate to 500 chars each, join with semicolon)
        topic_comments = '; '.join([str(content)[:500] for content in topic_docs['CONTENT']])
        
        topic_agg.append({
            'topic': topic,
            'n': len(topic_docs),
            'n_char': len(topic_comments),
            'topic_comments': topic_comments,
            'representation': representation
        })
    
    topic_agg_df = pd.DataFrame(topic_agg)
    
    # Generate descriptions using Cortex
    topic_descriptions = []
    
    for _, row in topic_agg_df.iterrows():
        # Construct prompt for three-part response
        prompt_text = f"""The following is a semi-colon separated list of comments from people impacted by recent wildfires in Los Angeles county (the Eaton Fire in Altadena and the Palisades fire in Pacific Palisades). These comments contain suggestions or proposals for government action. The keywords associated with these comments and this topic are: {row['representation']}.

Please analyze these comments and provide three things separated by the ^ character:

1) A 3-7 word topic name phrased as an imperative statement starting with a verb (e.g., "Bury power lines underground", "Provide legal aid services", "Create evacuation alternatives")

2) A 50-100 word, plain english, neutral perspective summary that explains the general problem being addressed, the objective to be achieved, and the value and impact that would be created for a lay reader. Have a neutral tone (similar to a citizen voting guide), and be sure to succinctly include the concerns or tradeoffs that opponents might raise

3) A succinct comma-separated list of representative actions that would implement this policy (e.g., if the topic is expand affordable housing, actions could be: Build public housing, Subsidize housing costs, Increase affordable units, Support land trusts)

Format your response exactly as: [TOPIC NAME] ^ [SUMMARY] ^ [ACTIONS]

<comments>{row['topic_comments']}</comments>"""
        
        prompt = [{
            "role": "user",
            "content": prompt_text
        }]
        
        options = CompleteOptions(
            max_tokens=500,  # Increased for longer response
            temperature=0.3,  # Lower temperature for more consistent results
            top_p=1,
            guardrails=False
        )
        
        try:
            result = complete(
                model="claude-3-5-sonnet",
                # model="snowflake-llama-3.1-405b",
                prompt=prompt,
                session=session,
                stream=False,  # Set to False for simpler handling
                options=options
            )
            
            # Extract and parse the three-part result
            if isinstance(result, str):
                response = result.strip()
            else:
                response = "".join(result).strip()
            
            # Parse the three components separated by ^
            parts = response.split('^')
            if len(parts) == 3:
                topic_name = parts[0].strip()
                summary = parts[1].strip()
                actions = parts[2].strip()
            else:
                # Fallback if parsing fails
                print(f"Warning: Could not parse response for topic {row['topic']}: {response}")
                topic_name = f"Address Topic {row['topic']} Issues"
                summary = "Summary not available"
                actions = "Actions not available"
                
        except Exception as e:
            print(f"Error generating description for topic {row['topic']}: {e}")
            topic_name = f"Address Topic {row['topic']} Issues"
            summary = "Summary not available"
            actions = "Actions not available"
        
        topic_descriptions.append({
            'topic': row['topic'],
            'n': row['n'],
            'topic_name': topic_name,
            'summary': summary,
            'actions': actions
        })
    
    return pd.DataFrame(topic_descriptions)



# # Option 1: Test on specific topic numbers
# test_topics = [0, 1, 2]  # Replace with actual topic numbers you want to test
# test_doc_df = doc_df[doc_df['TOPIC'].isin(test_topics)].copy()

# print(f"Testing on topics: {test_topics}")
# print(f"Documents in test set: {len(test_doc_df)}")

# # Run the function on test data
# test_descriptions = generate_topic_descriptions(test_doc_df, topic_info, topic_model, session)
# print("\nTest Results:")
# for _, row in test_descriptions.iterrows():
#     print(f"\nTopic: {row['topic']} | Count: {row['n']}")
#     print(f"  Name: {row['topic_name']}")
#     print(f"  Summary: {row['summary']}")
#     print(f"  Actions: {row['actions']}")


    
# Generate topic descriptions
print("Generating topic descriptions...")
topic_descriptions_df = generate_topic_descriptions(doc_df, topic_info, topic_model, session)

# Display results
print("\nGenerated Topic Descriptions:")
for _, row in topic_descriptions_df.iterrows():
    print(f"\nTopic: {row['topic']} | Count: {row['n']}")
    print(f"  Name: {row['topic_name']}")
    print(f"  Summary: {row['summary']}")
    print(f"  Actions: {row['actions']}")

# Add the descriptions back to your existing dataframes if needed
# Merge with topic_info
topic_info_with_desc = topic_info.merge(
    topic_descriptions_df[['topic', 'topic_name', 'summary', 'actions']], 
    left_on='Topic', 
    right_on='topic', 
    how='left'
)



## Persist topic-model results to Snowflake  

1. **Assemble final DataFrames**  
   * `doc_df` – comment-level rows (`COMMENT_ID`, `OPTION`, `topic`, `content`, `target`).  
   * `doc_df_with_descriptions` – adds the AI-generated `topic_name / summary / actions`.  
   * `final_df` – joins those fields back to the original embedding table for a one-stop view.

2. **Write back to Snowflake**  
   * `OPTION_TOPIC_MODELING` – every comment + its topic labels and descriptions.  
   * `OPTION_TOPIC_SUMMARY`  – one row per topic with counts, name, summary, actions.  
   Both tables are created (or replaced) via `session.write_pandas(..., overwrite=True)`.

3. **Return `final_df`** so you can inspect the merged result in-notebook.


In [ ]:
# Update doc_df creation to match the original dataframe column names
doc_df = pd.DataFrame({
    'COMMENT_ID': cmnt_embeddings_df['COMMENT_ID'],  # lowercase to match original
    'OPTION': docs,  # This is the OPTION column that became SUMMARY
    'topic': topics,
    'content': cmnt_embeddings_df['CONTENT'],  # lowercase to match original
    'target': cmnt_embeddings_df['TARGET']  # include target from original
})



# Merge topic descriptions with doc_df
doc_df_with_descriptions = doc_df.merge(
    topic_descriptions_df[['topic', 'topic_name', 'summary', 'actions']], 
    on='topic', 
    how='left'
)

# Merge back with original cmnt_embeddings_df to get all original columns
final_df = cmnt_embeddings_df.merge(
    doc_df_with_descriptions[['COMMENT_ID','OPTION', 'topic', 'topic_name', 'summary', 'actions']], 
    on=['COMMENT_ID', 'OPTION'], 
    how='left'
)

session.write_pandas(
    final_df,
    table_name="OPTION_TOPIC_MODELING",
    database="TRANSFORM_ENGCA_DEV",
    schema="DBT_MMARKS_ETHELO",
    overwrite=True,
    quote_identifiers=False,
    auto_create_table=True
)

session.write_pandas(
    topic_descriptions_df,
    table_name="OPTION_TOPIC_SUMMARY",
    database="TRANSFORM_ENGCA_DEV",
    schema="DBT_MMARKS_ETHELO",
    overwrite=True,
    quote_identifiers=False,
    auto_create_table=True
)

final_df